In [ ]:
import mg5qs_imports as qs
from pathlib import Path
import os
import numpy as np

In [ ]:
INPUT_PATH = Path.cwd()/'mg5' # madgraph cards
# note that for this example, we move to a process which directly generates taus
qs.edit_card(INPUT_PATH, card_name='proc_card_example4.dat')

In [ ]:
output_name, FRAMEWORK_PATH = qs.run_MG5(INPUT_PATH, proc_card_name='proc_card_example4.dat')

In [ ]:
qs.edit_card(FRAMEWORK_PATH)

In [ ]:
Wplus_id = 24
tau_id = 15
card = qs.ParamCard(FRAMEWORK_PATH / 'Cards' / 'param_card.dat')
card.get_value('MASS', Wplus_id), card.get_value('MASS', tau_id)

In [ ]:
# want to vary both masses in a range where the physics is highly sensitive
MTAU = [1.777, 40, 80, 120, 160]
MW = [40, 80.419, 120, 160, 200] 
MTAU, MW

In [ ]:
card = qs.ParamCard(FRAMEWORK_PATH / 'Cards' / 'param_card.dat')
for mtau in MTAU:
    card.set_value('MASS', tau_id, mtau)
    for mw in MW:
        card.set_value('MASS', Wplus_id, mw)
        qs.generate_LHE(card, FRAMEWORK_PATH)

In [ ]:
qs.pythia_parallel([tau_id], FRAMEWORK_PATH, 'EXAMPLE_CONTOUR', topics='P_mu', size=1000000)

In [ ]:
data = qs.unpickle('EXAMPLE_CONTOUR', concat=False)

In [ ]:
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
for v in data.values(): # compute pT col in each dataframe 
    v[1]['pT'] = np.sqrt((v[1]['px']**2)+(v[1]['py']**2))

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(16, 12))
axes = axes.flatten()
BINS = np.linspace(0, 500, 30)

for idx, pc_df in enumerate(data.values()):
    ax = axes[idx]
    ax.hist(pc_df[1]['pT'], bins=BINS)
    m_tau = pc_df[0].get_value('MASS', 15)
    m_W = pc_df[0].get_value('MASS', 24)
    ax.set_title(f"$m_\\tau$ = {m_tau},   $m_W$ = {m_W}")
    ax.set_xlabel('GeV')
    ax.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
for k in data.keys(): # find SM run by checking parameters 
    if np.isclose(data[k][0].get_value('MASS', tau_id), 1.777) and np.isclose(data[k][0].get_value('MASS', Wplus_id), 80.419):
        SM = k
SM

In [ ]:
contour = []
for v in data.values(): #construct contour plot data using KS test
    x = v[0].get_value('MASS', tau_id)
    y = v[0].get_value('MASS', Wplus_id)
    z = stats.ks_2samp(v[1]['pT'], data[SM][1]['pT']).pvalue
    contour.append([x, y, z])
contour = np.array(contour)
contour

In [ ]:
x = contour[:, 0]
y = contour[:, 1]
z = contour[:, 2]

plt.figure(figsize=(8, 6))
cs = plt.tricontour(x, y, z, levels=[0.05], colors='red')
plt.clabel(cs, inline=True, fontsize=10)
sc = plt.scatter(x, y, c=z, cmap='viridis', edgecolors='k', s=500)
plt.colorbar(sc, label='p value')
plt.xlabel('$m_\\tau$', fontsize=18)
plt.ylabel('$m_{W^+}$', fontsize=18)
plt.title('Contour over $\\tau$ and $W^+$ masses')
plt.grid(True)
plt.show()